# Google Books Ngram Explorer

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

Advancing AI Anthropology through computational approaches to qualitative research.

<br>

---

<br>

## What This Notebook Does

This notebook provides programmatic access to Google Books Ngram data, enabling researchers to analyze historical word frequency patterns across massive text corpora. Rather than manually downloading data from the web interface, this notebook fetches ngram data, visualizes trends, and exports results in formats structured for computational analysis.

Google Books Ngram data tracks the relative frequency of words and phrases across millions of digitized books from 1800 to 2022. This data can help researchers trace how concepts, terms, and ideas have risen or fallen in published discourse over more than two centuries.

## Key Features

1. **Multiple Corpus Selection**: Access data from 11 corpora including English (2019), American English, British English, and 8 other languages
2. **Flexible Time Ranges**: Query any date range from 1800 to 2022
3. **Case-Insensitive Searches**: Combine case variations for more accurate frequency analysis
4. **Smoothing Controls**: Apply moving average smoothing (0–50 years) to reduce noise in temporal patterns
5. **Multiple Unit Formats**: Display results as raw frequencies, per million, per billion, or percentage
6. **Structured Export**: Download results as CSV or Excel for further analysis
7. **Interactive Visualization**: Built-in plotting with branded styling

## Workflow

1. **Configure**: Set search terms, date range, corpus, and analysis options
2. **Fetch & Visualize**: Retrieve data from Google Books and generate a visualization
3. **Review**: Examine the plot and data preview
4. **Export**: Download structured data for use in other tools

## Applications

This notebook supports research that benefits from understanding how language and concepts change over time — tracing the historical trajectory of disciplinary terms, identifying shifts in public discourse, and contextualizing qualitative findings within long-term linguistic trends. Useful for dissertation research, digital humanities projects, and any study where historical text patterns matter.

## Methodological Positioning

This notebook employs **digital methods** — leveraging large-scale digital infrastructures to examine cultural patterns through published text. Google Books Ngram data reflects the contents of digitized books, not all published material, and should be interpreted with awareness of the corpus's known biases (over-representation of scientific texts, English-language dominance, OCR errors in older material).

**Important**: This notebook retrieves data but does not interpret it. Interpretation remains the work of the researcher.

## Target Audience

Designed for anthropologists and qualitative researchers working with historical text analysis — from graduate students exploring how disciplinary terms have evolved to research teams contextualizing fieldwork within broader discursive trends.

## Technical Approach

The notebook interfaces directly with the Google Books Ngram API, handling parameter formatting and response parsing. It implements retry logic with exponential backoff for reliability and provides both visual and tabular output formats.

## Contributing to AI Anthropology

This notebook contributes to the emerging field of AI Anthropology — which combines studying AI as cultural artifact, using AI to enhance ethnographic research, and applying anthropological insights to AI development (Artz, forthcoming). By open-sourcing these tools, this work advances the collective capacity of anthropologists to work effectively with computational methods.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## References

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

Artz, Matt. Forthcoming. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

## Setup and Installation

*Install required Python packages and import necessary libraries for API interaction, data processing, and visualization. Run this cell first to ensure all dependencies are available.*

In [2]:
# Install required packages
!pip install ipywidgets requests pandas matplotlib openpyxl -q

import json
import time
import random
from typing import Dict, List

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
import ipywidgets as widgets

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("\u2713 Libraries imported successfully")
print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")

✓ Libraries imported successfully
✓ Environment: Google Colab


## Configuration and Helper Functions

*Define corpus options, default parameters, and core functions for API communication, data transformation, and unit conversions. These functions handle the technical details of interacting with the Google Books Ngram service.*

In [9]:
# Available corpora from Google Books Ngram Viewer (2019 editions)
CORPORA_2019: Dict[str, int] = {
    "English (2019)": 26,
    "English Fiction (2019)": 27,
    "American English (2019)": 28,
    "British English (2019)": 29,
    "French (2019)": 30,
    "German (2019)": 31,
    "Spanish (2019)": 32,
    "Italian (2019)": 33,
    "Chinese Simplified (2019)": 34,
    "Hebrew (2019)": 35,
    "Russian (2019)": 36,
}

# Default parameters for searches
DEFAULTS = {
    "content": " ",
    "year_start": 1800,
    "year_end": 2022,
    "corpus": 26,
    "smoothing": 3,
    "case_insensitive": False,
    "unit": "per million",
    "round_decimals": 6,
}

NGRAM_ENDPOINT = "https://books.google.com/ngrams/json"


def _ua() -> str:
    """Generate a randomized user agent string for API requests."""
    base = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/"
    ver = f"{random.randint(100, 125)}.0.{random.randint(1200, 6000)}.{random.randint(10, 150)}"
    return f"{base}{ver} Safari/537.36"


def build_params(content: str,
                 year_start: int,
                 year_end: int,
                 corpus: int,
                 smoothing: int,
                 case_insensitive: bool) -> Dict:
    """Construct parameters dictionary for API request."""
    return {
        "content": content.strip(),
        "year_start": int(year_start),
        "year_end": int(year_end),
        "corpus": int(corpus),
        "smoothing": int(smoothing),
        "case_insensitive": bool(case_insensitive),
    }


def fetch_ngrams(params: Dict,
                 timeout: int = 30,
                 retries: int = 2,
                 backoff: float = 1.5) -> List[Dict]:
    """
    Fetch ngram data from Google Books API with retry logic.

    Args:
        params: Query parameters dictionary
        timeout: Request timeout in seconds
        retries: Number of retry attempts
        backoff: Exponential backoff multiplier

    Returns:
        List of ngram data dictionaries
    """
    headers = {"User-Agent": _ua()}
    attempt = 0
    while True:
        try:
            r = requests.get(NGRAM_ENDPOINT, params=params, headers=headers, timeout=timeout)
            r.raise_for_status()
            data = r.json()
            if not isinstance(data, list):
                raise ValueError("Unexpected response structure (expected a JSON list).")
            return data
        except Exception as e:
            attempt += 1
            if attempt > retries:
                raise
            time.sleep(backoff ** attempt)


def json_to_wide_dataframe(payload: List[Dict], year_start: int, year_end: int) -> pd.DataFrame:
    """
    Transform API response into wide-format DataFrame.

    Args:
        payload: List of ngram data from API
        year_start: Starting year for data
        year_end: Ending year for data

    Returns:
        DataFrame with years as rows and ngrams as columns
    """
    years = list(range(int(year_start), int(year_end) + 1))
    df = pd.DataFrame({"year": years}).set_index("year")

    for entry in payload:
        label = entry.get("ngram")
        etype = (entry.get("type") or "").lower()
        series = entry.get("timeseries", [])

        # Clean up case-insensitive labels
        if etype == "caseinsensitive" and label and label.lower().endswith("(all)"):
            label = label[:-5].strip()

        if not label or not isinstance(series, list):
            continue

        # Ensure series matches year range
        if len(series) != len(years):
            series = (series + [None] * len(years))[: len(years)]

        df[label] = series

    # Fallback if no columns were added
    if len(df.columns) == 0:
        for i, entry in enumerate(payload):
            col = entry.get("ngram") or f"series_{i+1}"
            series = entry.get("timeseries", [])
            if len(series) != len(years):
                series = (series + [None] * len(years))[: len(years)]
            df[col] = series

    return df.reset_index()


def unit_scale_and_label(unit: str):
    """Return scaling factor and y-axis label for given unit type."""
    if unit == "raw":
        return 1.0, "Relative frequency"
    if unit == "per million":
        return 1_000_000.0, "Occurrences per million tokens"
    if unit == "per billion":
        return 1_000_000_000.0, "Occurrences per billion tokens"
    if unit == "percent":
        return 100.0, "Percent of tokens"
    return 1.0, "Relative frequency"


def apply_units(df: pd.DataFrame, unit: str) -> pd.DataFrame:
    """Apply unit scaling to all numeric columns in DataFrame."""
    scale, _ = unit_scale_and_label(unit)
    out = df.copy()
    for c in out.columns:
        if c == "year":
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce") * scale
    return out


PLOT_COLORS = ['#274C77', '#6096BA', '#A3CEF1', '#8B8C89', '#4A7C59']

def plot_df(df: pd.DataFrame, unit: str):
    """Generate line plot of ngram frequencies over time."""
    _, ylabel = unit_scale_and_label(unit)
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#FAFAFA')
    x = df["year"].values
    for i, col in enumerate(c for c in df.columns if c != "year"):
        ax.plot(x, df[col].values, label=col, linewidth=2,
                color=PLOT_COLORS[i % len(PLOT_COLORS)])
    ax.legend(loc="best", frameon=True, facecolor='white', edgecolor='#E7ECEF')
    ax.set_title("Google Books Ngram Viewer", fontsize=14, fontweight='bold',
                 color='#274C77', pad=15)
    ax.set_xlabel("Year", fontsize=11, color='#274C77')
    ax.set_ylabel(ylabel, fontsize=11, color='#274C77')
    ax.grid(True, alpha=0.3, color='#8B8C89')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#E7ECEF')
    ax.spines['bottom'].set_color('#E7ECEF')
    plt.tight_layout()
    plt.show()

print("✓ Helper functions defined")

✓ Helper functions defined


## Interactive Search Interface

*Configure your ngram search using the controls below, then fetch and visualize the data. The interface provides options for search terms, date ranges, corpus selection, and output formatting.*

In [ ]:
# Color palette for consistent styling
COLORS = {
    'primary_bg': '#E7ECEF',
    'primary_text': '#274C77',
    'interactive': '#6096BA',
    'secondary_bg': '#A3CEF1',
    'neutral': '#8B8C89'
}

# Create styled widgets
query_w = widgets.Textarea(
    value=DEFAULTS["content"],
    description="Query",
    placeholder="Comma-separated phrases (supports wildcards, POS tags, arithmetic)",
    layout=widgets.Layout(width="100%", height="70px"),
    style={'description_width': '80px'}
)

year_w = widgets.IntRangeSlider(
    value=(DEFAULTS["year_start"], DEFAULTS["year_end"]),
    min=1800, max=2022, step=1,
    description="Years",
    continuous_update=False,
    readout=True,
    style={'description_width': '80px'}
)

corpus_w = widgets.Dropdown(
    options=[(k, v) for k, v in CORPORA_2019.items()],
    value=DEFAULTS["corpus"],
    description="Corpus",
    style={'description_width': '80px'}
)

case_w = widgets.Checkbox(
    value=DEFAULTS["case_insensitive"],
    description="Case-insensitive",
    indent=False,
    style={'description_width': '120px'}
)

smoothing_w = widgets.IntSlider(
    value=DEFAULTS["smoothing"],
    min=0, max=50, step=1,
    description="Smoothing",
    style={'description_width': '80px'}
)

unit_w = widgets.Dropdown(
    options=["raw", "per million", "per billion", "percent"],
    value=DEFAULTS["unit"],
    description="Units",
    style={'description_width': '80px'}
)

# Create action buttons
fetch_btn = widgets.Button(
    description="\U0001f50d Fetch & Plot",
    button_style="info",
    tooltip="Retrieve data and generate visualization",
    layout=widgets.Layout(width='180px')
)

csv_btn = widgets.Button(
    description="\U0001f4ca Export CSV",
    button_style="",
    tooltip="Download CSV with full precision numbers",
    layout=widgets.Layout(width='180px')
)

xlsx_btn = widgets.Button(
    description="\U0001f4cb Export Excel",
    button_style="",
    tooltip="Download styled Excel workbook",
    layout=widgets.Layout(width='180px')
)

# Output area
out = widgets.Output()

# Global variables to store results
last_df = None
last_df_units = None
last_csv_path = None


# Create help text
help_html = widgets.HTML(
    value=f"""
    <div style="background-color: {COLORS['primary_bg']};
                padding: 15px;
                border-radius: 10px;
                border-left: 5px solid {COLORS['primary_text']};
                margin: 10px 0;">
        <h3 style="color: {COLORS['primary_text']}; margin-top: 0;">\U0001f4d6 Query Help</h3>
        <p style="margin: 5px 0;"><strong>Basic search:</strong> Enter words or phrases separated by commas</p>
        <p style="margin: 5px 0;"><strong>Wildcards:</strong> Use * for any characters (e.g., "anthro*")</p>
        <p style="margin: 5px 0;"><strong>POS tags:</strong> Add _NOUN, _VERB, etc. (e.g., "culture_NOUN")</p>
        <p style="margin: 5px 0;"><strong>Operators:</strong> Use + - * / for arithmetic (e.g., "digital + computation")</p>
    </div>
    """
)

print("\u2713 Interface widgets created")

In [ ]:
def _make_filename_base():
    """Generate a safe filename base from current query settings."""
    content = query_w.value.strip()
    ys, ye = year_w.value
    safe_stub = "".join(ch for ch in content.replace(",", "_") if ch.isalnum() or ch in ("_", "-"))
    if not safe_stub:
        safe_stub = "ngrams"
    safe_stub = safe_stub[:32]
    unit_str = unit_w.value.replace(' ', '')
    return f"{safe_stub}_{ys}-{ye}_{unit_str}"


def on_fetch_clicked(_):
    """Handle fetch button click event."""
    global last_df, last_df_units, last_csv_path
    out.clear_output()

    with out:
        content = query_w.value.strip()
        ys, ye = year_w.value

        if not content:
            display(Markdown("\u26a0\ufe0f **Please enter a search query**"))
            return

        params = build_params(
            content=content,
            year_start=ys,
            year_end=ye,
            corpus=int(corpus_w.value),
            smoothing=int(smoothing_w.value),
            case_insensitive=bool(case_w.value),
        )

        display(Markdown("### \U0001f504 Fetching Data..."))
        display(Markdown(f"**Endpoint:** `{NGRAM_ENDPOINT}`"))
        display(Markdown(f"**Parameters:** `{json.dumps(params, indent=2)}`"))

        try:
            payload = fetch_ngrams(params, retries=2, backoff=1.6, timeout=30)
            display(Markdown(f"**\u2713 Retrieved {len(payload)} ngram series**"))

            if not payload:
                display(Markdown("\u26a0\ufe0f **No data returned.** Try adjusting your search terms or date range."))
                return

            df = json_to_wide_dataframe(payload, ys, ye)
            df_units = apply_units(df, unit_w.value)

            display(Markdown("### \U0001f4ca Data Preview"))
            display(df_units.head(10))

            display(Markdown("### \U0001f4c8 Visualization"))

            # Create and display the plot
            _, ylabel = unit_scale_and_label(unit_w.value)
            fig, ax = plt.subplots(figsize=(12, 6))
            fig.patch.set_facecolor('#FFFFFF')
            ax.set_facecolor('#FAFAFA')
            x = df_units["year"].values
            for i, col in enumerate(c for c in df_units.columns if c != "year"):
                ax.plot(x, df_units[col].values, label=col, linewidth=2,
                        color=PLOT_COLORS[i % len(PLOT_COLORS)])
            ax.legend(loc="best", fontsize=10, frameon=True, facecolor='white', edgecolor='#E7ECEF')
            ax.set_title(f"Google Books Ngram: {content}", fontsize=14, fontweight='bold',
                         color='#274C77', pad=15)
            ax.set_xlabel("Year", fontsize=11, color='#274C77')
            ax.set_ylabel(ylabel, fontsize=11, color='#274C77')
            ax.grid(True, alpha=0.3, color='#8B8C89')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('#E7ECEF')
            ax.spines['bottom'].set_color('#E7ECEF')
            plt.tight_layout()

            base = _make_filename_base()

            # Save chart
            chart_path = f"{base}_chart.png"
            plt.savefig(chart_path, dpi=300, bbox_inches='tight')
            plt.show()

            # Save CSV
            csv_path = f"{base}.csv"
            df_units.to_csv(csv_path, index=False)

            last_df = df
            last_df_units = df_units
            last_csv_path = csv_path

            display(Markdown(f"\u2713 **Data saved:** `{csv_path}`"))
            display(Markdown(f"\u2713 **Chart saved:** `{chart_path}`"))

        except Exception as e:
            display(Markdown(f"\u274c **Error:** {str(e)}"))
            display(Markdown("*Check your internet connection and try again.*"))


def on_csv_clicked(_):
    """Handle CSV download button click."""
    if not last_csv_path:
        with out:
            display(Markdown("\u26a0\ufe0f **No data to download.** Click 'Fetch & Plot' first."))
        return

    if IN_COLAB:
        colab_files.download(last_csv_path)
    else:
        with out:
            display(Markdown(f"\u2713 **CSV saved locally:** `{last_csv_path}`"))


def on_xlsx_clicked(_):
    """Handle Excel download button click."""
    if last_df_units is None:
        with out:
            display(Markdown("\u26a0\ufe0f **No data to download.** Click 'Fetch & Plot' first."))
        return

    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

    base = _make_filename_base()
    filepath = f"{base}.xlsx"

    last_df_units.to_excel(filepath, sheet_name='Ngram Data', index=False, engine='openpyxl')

    # Style the Excel output
    wb = load_workbook(filepath)
    header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    thin_border = Border(
        left=Side(style='thin', color='E7ECEF'),
        right=Side(style='thin', color='E7ECEF'),
        top=Side(style='thin', color='E7ECEF'),
        bottom=Side(style='thin', color='E7ECEF'),
    )

    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = header_alignment
            cell.border = thin_border

        for col in ws.columns:
            max_length = max(len(str(cell.value or '')) for cell in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_length + 2, 10), 60)

        ws.freeze_panes = 'A2'

        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
                cell.border = thin_border

    wb.save(filepath)

    if IN_COLAB:
        colab_files.download(filepath)
        with out:
            display(Markdown(f"\U0001f4e5 **Downloaded:** `{filepath}`"))
    else:
        with out:
            display(Markdown(f"\u2713 **Excel saved locally:** `{filepath}`"))


# Attach event handlers
fetch_btn.on_click(on_fetch_clicked)
csv_btn.on_click(on_csv_clicked)
xlsx_btn.on_click(on_xlsx_clicked)

print("\u2713 Event handlers configured")

In [ ]:
# Assemble and display the interface
ui = widgets.VBox([
    widgets.HTML(
        value="""
        <div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px;
                    margin-bottom: 20px; border-left: 5px solid #274C77;">
            <h2 style="color: #274C77; margin-top: 0;">\U0001f4da Google Books Ngram Search</h2>
            <p>Enter search terms, configure options, then click <strong>Fetch & Plot</strong> to retrieve and visualize ngram data.</p>
        </div>
        """
    ),
    help_html,
    query_w,
    widgets.HBox([year_w], layout=widgets.Layout(margin='10px 0')),
    widgets.HBox([corpus_w, case_w, smoothing_w], layout=widgets.Layout(margin='10px 0')),
    widgets.HBox([unit_w], layout=widgets.Layout(margin='10px 0')),
    widgets.HBox([fetch_btn, csv_btn, xlsx_btn],
                 layout=widgets.Layout(margin='20px 0', justify_content='center')),
    out
], layout=widgets.Layout(padding='20px'))

display(ui)